# Step 0: Extract Image-Caption Pairs

Extract valid (image, caption) pairs from crawled ViFactCheck articles.
COOLANT uses these pairs where matched = Real, unmatched = Fake.

Prefers recrawled data (better captions) if available, falls back to cleaned data.

- Input: `../data/json/recrawled/*_recrawled.json` or `../data/json/*_cleaned.json`
- Images: `../data/jpg/`
- Output: `./pairs/pairs_{train,dev,test}.json`

Prerequisites: Optionally run `00_recrawl_captions.ipynb` first for better captions.

In [1]:
import sys
import importlib
from pathlib import Path

project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

# Force reload to pick up latest code
import src.processing.coolant.pair_extractor as pe_mod
importlib.reload(pe_mod)

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [2]:
from src.processing.coolant.pair_extractor import PairExtractor, clean_caption, is_credit_only
import os

# Paths
JSON_DIR = project_root / "notebooks" / "data" / "json"
RECRAWLED_DIR = JSON_DIR / "recrawled"
JPG_DIR = project_root / "notebooks" / "data" / "jpg"
OUTPUT_DIR = Path("./pairs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Use recrawled data if available, otherwise cleaned
if RECRAWLED_DIR.exists() and any(RECRAWLED_DIR.glob("*_recrawled.json")):
    source_dir = RECRAWLED_DIR
    file_suffix = "_recrawled"
    print(f"Using RECRAWLED data from: {RECRAWLED_DIR}")
else:
    source_dir = JSON_DIR
    file_suffix = "_cleaned"
    print(f"Using CLEANED data from: {JSON_DIR}")

# Extract pairs (min 5 chars caption, with text cleaning + credit filtering)
extractor = PairExtractor(jpg_base_dir=str(JPG_DIR), min_caption_len=5)

all_pairs = {}
for split in ["train", "dev", "test"]:
    # Try recrawled first, then cleaned, then raw
    json_path = source_dir / f"news_data_vifactcheck_{split}{file_suffix}.json"
    if not json_path.exists():
        json_path = JSON_DIR / f"news_data_vifactcheck_{split}_cleaned.json"
    if not json_path.exists():
        json_path = JSON_DIR / f"news_data_vifactcheck_{split}.json"
    if json_path.exists():
        print(f"\n{split}: loading from {json_path.name}")
        pairs = extractor.extract_from_json(str(json_path))
        all_pairs[split] = pairs
    else:
        print(f"\n{split}: not found")

print("\n" + "=" * 50)
print("EXTRACTION RESULTS")
print("=" * 50)
for split, pairs in all_pairs.items():
    extractor.print_stats(pairs, split)

# Save
for split, pairs in all_pairs.items():
    extractor.save_pairs(pairs, str(OUTPUT_DIR / f"pairs_{split}.json"))

print(f"\nDone. Files in {OUTPUT_DIR.resolve()}")

Using RECRAWLED data from: g:\My Drive\Thesis_Final\fake-new-detection\notebooks\data\json\recrawled

train: loading from news_data_vifactcheck_train_recrawled.json
    Total images: 4164, Valid pairs: 3537
    Skipped: {'no_caption': 75, 'credit_only': 0, 'too_short': 0, 'no_image': 552}

dev: loading from news_data_vifactcheck_dev_recrawled.json
    Total images: 1226, Valid pairs: 1054
    Skipped: {'no_caption': 14, 'credit_only': 0, 'too_short': 0, 'no_image': 158}

test: loading from news_data_vifactcheck_test_recrawled.json
    Total images: 1033, Valid pairs: 876
    Skipped: {'no_caption': 16, 'credit_only': 0, 'too_short': 0, 'no_image': 141}

EXTRACTION RESULTS
  train: 3537 pairs from 1653 articles
    Caption length: avg=86, min=11, max=545
  dev: 1054 pairs from 519 articles
    Caption length: avg=80, min=11, max=426
  test: 876 pairs from 412 articles
    Caption length: avg=86, min=17, max=426
  Saved 3537 pairs to pairs\pairs_train.json
  Saved 1054 pairs to pairs\pai

In [3]:
# Show sample pairs with cleaned captions
if all_pairs.get("train"):
    print("Sample valid pairs (train):")
    for p in all_pairs["train"][:5]:
        print(f"  Image: {p['folder_path']}")
        print(f"  Caption: {p['caption'][:120]}")
        print(f"  Article: {p['article_idx']}")
        print()

    # Show what got cleaned/filtered
    print("Caption cleaning examples:")
    test_cases = [
        "Một góc dự án Ocean Park 3. Ảnh: NVCC",
        "Ảnh minh họa: Reuters",
        "TTXVN",
        "Bên trong xưởng sản xuất tiền giả - Ảnh 1.",
        "Phó Thủ tướng Trần Hồng Hà phát biểu (Nguồn: VGP)",
        "Ảnh by AFP",
        "Photo: Getty Images",
    ]
    for raw in test_cases:
        cleaned = clean_caption(raw)
        is_credit = is_credit_only(cleaned)
        status = "SKIP (credit only)" if is_credit else f"OK -> '{cleaned}'"
        print(f"  '{raw}' => {status}")

Sample valid pairs (train):
  Image: bao_chinh_phu/bao_chinh_phu_6d057d2da5.jpg
  Caption: Bệnh viện K đã tiến hành mổ bóc tách u nặng tới gần 10 kg cho bệnh nhân Lê Thị P. 65 tuổi quê tại Nam Định với chẩn đoán
  Article: 7

  Image: dan_tri/dan_tri_afcc41589d.jpg
  Caption: Có 21.714 người có tài sản ròng trên 30 triệu USD lựa chọn New York để cư trú chính hoặc sở hữu ngôi nhà thứ 2 tại đây
  Article: 9

  Image: vn_express/vn_express_b192eb7337.jpg
  Caption: Siêu Trái Đất Kepler-62f quay quanh ngôi sao nhỏ và mát hơn Mặt Trời ở cách 1.200 năm ánh sáng
  Article: 10

  Image: bao_chinh_phu/bao_chinh_phu_d9ff58958b.jpg
  Caption: Vietjet và chính quyền bang Queensland (Australia) vừa công bố sẽ khai thác đường bay thẳng đầu tiên kết nối Việt Nam và
  Article: 11

  Image: bao_chinh_phu/bao_chinh_phu_8bcaba5fa3.jpg
  Caption: Phó Phát ngôn Bộ Ngoại giao Phạm Thu Hằng
  Article: 12

Caption cleaning examples:
  'Một góc dự án Ocean Park 3. Ảnh: NVCC' => OK -> 'Một góc dự án Ocean Park 

In [4]:
# Coverage summary: how many images had valid captions vs skipped
import json

print("COVERAGE SUMMARY")
print("=" * 70)
print(f"{'Split':<8} {'Total Img':>10} {'Valid Pairs':>12} {'No Caption':>12} {'Credit Only':>12} {'%Valid':>8}")
print("-" * 70)

for split in ["train", "dev", "test"]:
    json_path = source_dir / f"news_data_vifactcheck_{split}{file_suffix}.json"
    if not json_path.exists():
        json_path = JSON_DIR / f"news_data_vifactcheck_{split}_cleaned.json"
    if not json_path.exists():
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        articles = json.load(f)

    total_imgs = sum(len(a.get("images", [])) for a in articles)
    n_pairs = len(all_pairs.get(split, []))
    pct = n_pairs / max(total_imgs, 1) * 100

    # Count skip reasons
    no_cap = 0
    credit = 0
    for a in articles:
        for img in a.get("images", []):
            raw = (img.get("caption") or "").strip()
            cleaned = clean_caption(raw)
            if not cleaned or cleaned == "Không được đề cập":
                no_cap += 1
            elif is_credit_only(cleaned) or len(cleaned) < 5:
                credit += 1

    print(f"{split:<8} {total_imgs:>10} {n_pairs:>12} {no_cap:>12} {credit:>12} {pct:>7.1f}%")

print(f"\nOnly pairs with BOTH a valid image AND a meaningful caption are included.")

COVERAGE SUMMARY
Split     Total Img  Valid Pairs   No Caption  Credit Only   %Valid
----------------------------------------------------------------------
train          4164         3537           84            0    84.9%
dev            1226         1054           20            0    86.0%
test           1033          876           18            0    84.8%

Only pairs with BOTH a valid image AND a meaningful caption are included.
